# Benchmark Workbench

We want to build a benchmark that measures something **new** about LLMs.

The loop:
1. Write a `@task` — a function that prompts an LLM and scores its response
2. Run it on several models to get scores  
3. Ask BenchPress: "can you predict these scores from existing benchmarks?"
4. If yes → our benchmark is redundant. If no → it measures something new.

## Setup

In [ ]:
import os
import hashlib
import numpy as np

from dotenv import load_dotenv
from cachy import enable_cachy

from evaluating_agi.benchpress import check_novelty, MODEL_IDX

enable_cachy()
load_dotenv()


True

The **kaggle-benchmarks SDK** gives us `@task` (decorator to define a benchmark)
and `llm.prompt()` (to call a model). It talks to any OpenAI-compatible API.

One gotcha: the SDK reads env vars at import time to create a default model handle.
So we set them first, then import.

In [ ]:
# SDK reads these at import time
os.environ["MODEL_PROXY_URL"]     = "https://api.openai.com/v1"
os.environ["MODEL_PROXY_API_KEY"] = os.environ["OPENAI_API_KEY"]
os.environ["LLM_DEFAULT"]         = "gpt-4.1-mini"

import kaggle_benchmarks as kb
from kaggle_benchmarks import assertions, llm, task
from kaggle_benchmarks.kaggle.model_proxy import ModelProxy

from pathlib import Path
kb.client.directory = Path("../results"); kb.client.directory.mkdir(exist_ok=True)
print(f"Default model: {llm.name}  |  results → {kb.client.directory.resolve()}")

Default model: gpt-4.1-mini  |  results → /Users/pengren/go/github.com/kafkasl/evaluating-agi/results


To run a task on **different** models, we create LLM handles via `ModelProxy`.

One gotcha: the SDK sends `seed=0` for reproducibility. OpenAI accepts it,
but Gemini and xAI reject it with a 400 error. So we patch it out for those.

In [ ]:
PROVIDERS = {
    "openai": ("https://api.openai.com/v1",                              "OPENAI_API_KEY"),
    "gemini": ("https://generativelanguage.googleapis.com/v1beta/openai", "GEMINI_API_KEY"),
    "xai":    ("https://api.x.ai/v1",                                    "XAI_API_KEY"),
}

def get_llm(provider, model):
    base_url, key_env = PROVIDERS[provider]
    m = ModelProxy(model=model, api_key=os.environ[key_env], base_url=base_url)
    if provider in ("gemini", "xai"):  # patch out seed
        _orig = m.invoke
        def patched(messages, system=None, **kw): kw.pop("seed", None); return _orig(messages, system=system, **kw)
        m.invoke = patched
    return m

## Models

For BenchPress to work, we need models that are **rows in its 83×49 matrix**.
Each entry maps: provider → API model name → BenchPress ID.

In [ ]:
EVAL_MODELS = [
    ("openai",  "gpt-4.1-mini",    "gpt-4.1-mini"),      # cheap
    ("gemini",  "gemini-2.5-flash", "gemini-2.5-flash"),  # cheap, different provider
    ("xai",     "grok-3-beta",      "grok-3-beta"),       # mid
    ("openai",  "gpt-4.1",          "gpt-4.1"),           # strong
    ("openai",  "o4-mini",          "o4-mini-high"),       # reasoning model
]

# Sanity check: all BenchPress IDs exist in the matrix
for _, _, bp_id in EVAL_MODELS: assert bp_id in MODEL_IDX, f"{bp_id} not in matrix!"
print(f"{len(EVAL_MODELS)} models, all in BenchPress matrix ✓")

5 models, all in BenchPress matrix ✓


## The pipeline

`run_on_models` runs a task on each model and returns scores in BenchPress format (0-100).
`evaluate_and_check` = run + novelty check in one call.

In [ ]:
def run_on_models(task_fn, models=None, n_trials=1):
    """Run a @task on each model → {benchpress_id: score_0_to_100}."""
    results = {}
    for provider, api_name, bp_id in (models or EVAL_MODELS):
        trial_scores = []
        for _ in range(n_trials):
            try:
                run = task_fn.run(get_llm(provider, api_name))
                ar = run.assertion_results
                s = sum(r.passed for r in ar) / len(ar) if ar else (1.0 if run.passed else 0.0)
            except Exception as e:
                print(f"  ✗ {api_name}: {e}"); s = 0.0
            trial_scores.append(s)
        avg = np.mean(trial_scores) * 100
        results[bp_id] = avg
        print(f"  {api_name:<25s} → {avg:5.1f}%  (n={len(trial_scores)})")
    return results

def evaluate_and_check(task_fn, name, models=None, n_trials=1):
    """Run on models → check novelty with BenchPress."""
    print(f"🔬 {name}")
    scores = run_on_models(task_fn, models, n_trials)
    return check_novelty(scores, name)

## Smoke test: a deliberately meaningless benchmark

Before building real benchmarks, lets validate the pipeline with a **random** one.

The task below asks 10 trivial questions (so we actually call the LLM), but the
score is a coin flip based on `md5(model_name + question_index)`. Each model gets
a different deterministic-but-meaningless score.

**What we expect**: BenchPress should *fail* to predict these scores (they have no
relationship to model capability), giving a high median error (~18+ pts).

In [ ]:
@task("random_benchmark", description="Scores by coin flip — should look like noise to BenchPress")
def random_benchmark(llm) -> None:
    questions = [
        "What is 2+2?", "Name a primary color.", "What planet is closest to the Sun?",
        "Is water wet?", "What comes after Monday?", "Spell the word cat.",
        "What is the boiling point of water in Celsius?", "Name a mammal.",
        "How many sides does a triangle have?", "What language is spoken in France?",
    ]
    for i, q in enumerate(questions):
        llm.prompt(q)  # actually call the LLM (we ignore the answer)
        h = int(hashlib.md5(f"{llm.name}_{i}".encode()).hexdigest(), 16)
        assertions.assert_true(h % 2 == 0, expectation=f"Coin flip for Q{i+1}: {q}")

In [ ]:
# Smoke test with all 5 models (3 is too few — LOO predicts from n-1, gets noisy)
r = evaluate_and_check(random_benchmark, "Random benchmark (expect: ~18 noise)", EVAL_MODELS)

🔬 Random benchmark (expect: ~18 noise)
  gpt-4.1-mini              →  60.0%  (n=1)
  gemini-2.5-flash          →  70.0%  (n=1)
  grok-3-beta               →  40.0%  (n=1)
  gpt-4.1                   →  70.0%  (n=1)
  o4-mini                   →  30.0%  (n=1)
  Model                        Actual   Pred   Err
  o4-mini (high)                 30.0   52.3  22.3 ← miss
  Gemini 2.5 Flash               70.0   49.2  20.8 ← miss
  GPT-4.1 mini                   60.0   74.7  14.7 ← miss
  Grok 3 Beta                    40.0   52.8  12.8 ← miss
  GPT-4.1                        70.0   57.4  12.6 ← miss
  Median LOO: 14.7 pts → NOVEL


In [ ]:
r

{'name': 'Random benchmark (expect: ~18 noise)',
 'median_error': np.float64(14.678091414598228),
 'results': [('gpt-4.1-mini',
   np.float64(60.0),
   np.float64(74.67809141459823),
   np.float64(14.678091414598228)),
  ('gemini-2.5-flash',
   np.float64(70.0),
   np.float64(49.213739386909424),
   np.float64(20.786260613090576)),
  ('grok-3-beta',
   np.float64(40.0),
   np.float64(52.773946409859555),
   np.float64(12.773946409859555)),
  ('gpt-4.1',
   np.float64(70.0),
   np.float64(57.437780785927295),
   np.float64(12.562219214072705)),
  ('o4-mini-high',
   np.float64(30.0),
   np.float64(52.30131363142786),
   np.float64(22.30131363142786))]}

## Control: a redundant benchmark

The random benchmark validated that BenchPress flags noise. Now the opposite: a **basic math**
benchmark that should be entirely predictable from existing math evals (MATH, AIME, GSM8K).

**Expected**: median LOO error < 5 pts (redundant).

In [ ]:
@task("basic_math", description="Grade-school to competition math — should be redundant with existing math evals")
def basic_math(llm) -> None:
    QA = [
        ("What is 17 * 23?", "391"),
        ("What is the derivative of x^3 + 2x?", "3x\\^2 \\+ 2"),
        ("Solve for x: 2x + 7 = 15", "4"),
        ("What is the sum of interior angles of a hexagon in degrees?", "720"),
        ("What is 144 / 12?", "12"),
        ("What is the GCD of 48 and 36?", "12"),
        ("If f(x) = 2x + 1, what is f(f(3))?", "15"),
        ("What is log base 2 of 64?", "6"),
        ("What is the 10th term of the arithmetic sequence 3, 7, 11, ...?", "39"),
        ("How many prime numbers are less than 20?", "8"),
    ]
    for q, pattern in QA:
        resp = llm.prompt(f"Answer with just the number/expression, nothing else.\n{q}")
        assertions.assert_contains_regex(pattern, resp, expectation=f"{q} → expected {pattern}")

In [ ]:
# Math benchmark — expect "redundant" (median LOO < 5)
evaluate_and_check(basic_math, "Basic math (expect: redundant)", EVAL_MODELS)

🔬 Basic math (expect: redundant)
  gpt-4.1-mini              →  90.0%  (n=1)
  gemini-2.5-flash          → 100.0%  (n=1)
  grok-3-beta               → 100.0%  (n=1)
  gpt-4.1                   → 100.0%  (n=1)
  o4-mini                   →  90.0%  (n=1)
  Model                        Actual   Pred   Err
  GPT-4.1 mini                   90.0   99.8   9.8
  o4-mini (high)                 90.0   99.8   9.8
  Grok 3 Beta                   100.0   97.9   2.1
  GPT-4.1                       100.0   98.0   2.0
  Gemini 2.5 Flash              100.0   98.2   1.8
  Median LOO: 2.1 pts → redundant


{'name': 'Basic math (expect: redundant)',
 'median_error': np.float64(2.052231972401316),
 'results': [('gpt-4.1-mini',
   np.float64(90.0),
   np.float64(99.80851832733836),
   np.float64(9.808518327338362)),
  ('gemini-2.5-flash',
   np.float64(100.0),
   np.float64(98.20582469950651),
   np.float64(1.7941753004934924)),
  ('grok-3-beta',
   np.float64(100.0),
   np.float64(97.94776802759868),
   np.float64(2.052231972401316)),
  ('gpt-4.1',
   np.float64(100.0),
   np.float64(97.95161945075486),
   np.float64(2.048380549245138)),
  ('o4-mini-high',
   np.float64(90.0),
   np.float64(99.79523597471471),
   np.float64(9.795235974714714))]}

## Reading the results

The report shows, for each model:
- **Actual**: the score our benchmark gave it
- **Pred**: what BenchPress predicted from the models other 49 benchmark scores
- **Error**: |Actual - Pred|

**Median LOO error** is the key number:
- **< 5** → redundant (BenchPress already knows this from existing benchmarks)
- **5–10** → partially novel
- **12+** → novel (measures something BenchPress cant predict)
- **~18+** → noise territory (no signal at all)

Our random benchmark should land in noise territory. If it does, the pipeline works
and we can trust it to tell us when a *real* benchmark is novel.